In [ ]:
%%sql


# EDA: MovieLens 100K / 1M

Notebook này phân tích khám phá dữ liệu (EDA) cho tập dữ liệu MovieLens 1M (hoặc 100K). 

> **Lưu ý cấu hình tự động (Project Standard):** 
> - Notebook đã được lập trình để tự động xác định đường dẫn đến thư mục chứa dữ liệu bất kể bạn chạy nó trong môi trường Local, Docker hay Google Colab.
> - Nếu chưa có dữ liệu, notebook sẽ **tự động tải trực tiếp từ máy chủ GroupLens và giải nén**, đảm bảo không xảy ra lỗi thiếu file khi giảng viên chạy thử.

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

def resolve_project_path(*parts):
    """Tìm đường dẫn tuyệt đối đến thư mục chứa file dữ liệu"""
    rel_path = Path(*parts)
    if rel_path.is_absolute() and rel_path.exists():
        return rel_path

    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for root in candidates:
        if (root / 'datasets').exists():
            path = root.joinpath(*parts)
            if path.exists():
                return path

    fallback = Path.cwd().joinpath(*parts)
    if fallback.exists():
        return fallback
    return rel_path

def download_and_extract_dataset(url: str, folder_name: str):
    """Tự động tải và giải nén tập dữ liệu nếu chưa tồn tại"""
    try:
        datasets_dir = resolve_project_path('datasets')
    except FileNotFoundError:
        # Nếu chưa tạo thư mục datasets thì tạo mới ở thư mục gốc của project
        datasets_dir = Path.cwd().parent / 'datasets' if (Path.cwd().parent / 'docker-compose.yml').exists() else Path.cwd() / 'datasets'
        datasets_dir.mkdir(parents=True, exist_ok=True)
        
    target_dir = datasets_dir / folder_name
    if target_dir.exists():
        print(f'==> Da tim thay du lieu tai: {target_dir}')
        return target_dir
        
    zip_path = datasets_dir / f'{folder_name}.zip'
    print(f'==> Khong tim thay {folder_name}. Dang tai tu GroupLens...')
    print(f'URL: {url}')
    urllib.request.urlretrieve(url, zip_path)
    
    print(f'==> Dang giai nen {zip_path.name}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(datasets_dir)
    zip_path.unlink() # Xóa file zip sau khi giải nén
    print(f'==> Da tai va giai nen thanh cong tai: {target_dir}')
    return target_dir

DATASET_ROOT = resolve_project_path('datasets')
IS_COLAB = False

print(f'DATASET_ROOT = {DATASET_ROOT}')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATASET = "ml-1m"  # Hoặc chọn "ml-100k"

def load_ratings(dataset: str) -> pd.DataFrame:
    dataset = dataset.strip().lower()
    if dataset == "ml-1m":
        # Tự động kiểm tra và tải dữ liệu nếu chưa có
        download_and_extract_dataset('https://files.grouplens.org/datasets/movielens/ml-1m.zip', 'ml-1m')
        path = resolve_project_path("datasets", "ml-1m", "ratings.dat")
        df = pd.read_csv(
            path,
            sep="::",
            engine="python",
            names=["user_id", "movie_id", "rating", "timestamp"],
        )
    elif dataset == "ml-100k":
        # Tự động kiểm tra và tải dữ liệu nếu chưa có
        download_and_extract_dataset('https://files.grouplens.org/datasets/movielens/ml-100k.zip', 'ml-100k')
        path = resolve_project_path("datasets", "ml-100k", "u.data")
        df = pd.read_csv(
            path,
            sep="\t",
            names=["user_id", "movie_id", "rating", "timestamp"],
        )
    else:
        raise ValueError("Dataset must be 'ml-1m' or 'ml-100k'.")
    return df

ratings = load_ratings(DATASET)
ratings.head()


In [ ]:
num_users = ratings["user_id"].nunique()
num_items = ratings["movie_id"].nunique()
num_ratings = len(ratings)
sparsity = 1 - num_ratings / (num_users * num_items)

print(f"Dataset: {DATASET}")
print(f"Users: {num_users}")
print(f"Items: {num_items}")
print(f"Ratings: {num_ratings}")
print(f"Sparsity: {sparsity:.4f}")

ratings["rating"].describe()


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(6, 4))
ratings["rating"].value_counts().sort_index().plot(kind="bar")
plt.title("Rating distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
user_counts = ratings.groupby("user_id").size()
item_counts = ratings.groupby("movie_id").size()

print("User interactions summary")
display(user_counts.describe())
print("Item interactions summary")
display(item_counts.describe())

plt.figure(figsize=(6, 4))
user_counts.plot(kind="hist", bins=50)
plt.title("Interactions per user")
plt.xlabel("Interactions")
plt.ylabel("Users")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
item_counts.plot(kind="hist", bins=50)
plt.title("Interactions per item")
plt.xlabel("Interactions")
plt.ylabel("Items")
plt.tight_layout()
plt.show()


In [ ]:
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings["year_month"] = ratings["datetime"].dt.to_period("M").astype(str)
monthly_counts = ratings.groupby("year_month").size()

plt.figure(figsize=(10, 4))
monthly_counts.plot()
plt.title("Ratings over time")
plt.xlabel("Year-Month")
plt.ylabel("Ratings")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
top_users = user_counts.sort_values(ascending=False).head(10)
top_items = item_counts.sort_values(ascending=False).head(10)

print("Top 10 users by interactions")
display(top_users)
print("Top 10 items by interactions")
display(top_items)
